# `EEGNet` 디코더 input/output 탐색 + 사전학습 체크포인트로 추론하기


## 0. 데이터 준비 + `EEGNet` 디코더 구조

[01_explore_eeg_dataset.ipynb](01_explore_eeg_dataset.ipynb)에서 살펴본 것과 같은 데이터를 준비합니다. `n2o.signal.dataset.DatasetLoader(name="BNCI2014_001").read()`는 subject 1의 raw 레코딩만 돌려주고, 전처리/윈도잉은 디코더가 합니다 — `BraindecodeDecoder`를 먼저 만들어서 `windowing_kwargs`를 넘겨두고, `decoder.prepare(raw_dataset)` 한 번으로 밴드패스+표준화와 이벤트 기준 윈도잉을 자동으로 적용합니다(무슨 일이 일어나는지는 01 참고).

 `n_chans`/`n_outputs`는 `info()` 메타데이터에서, `n_times`(모델 레이어를 만들려면 미리 알아야 하는 윈도우 길이)는 `n2o.decoder.expected_window_samples()`로 — 실제로 윈도잉을 해보기 전에, `window_by_event()`가 내부적으로 쓰는 것과 같은 두 값(트라이얼 자체의 annotation duration, sfreq)만으로 그 결과 길이를 미리 계산합니다.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import torch
from IPython.display import Markdown, display

from n2o.decoder import (
    BraindecodeDecoder,
    expected_window_samples,
    label_names,
)
from n2o.signal.dataset import DatasetLoader

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

# 아래 2번 섹션의 사전학습 체크포인트가 학습될 때 쓴 것과 같은 오프셋 -- 이렇게 한 곳에서만
# 정의해두면 노트북 전체에서 서로 다른 윈도잉 설정을 두 번 반복할 필요가 없습니다.
WINDOWING_KWARGS = {"start_offset_sec": -0.5, "stop_offset_sec": 0.0}

loader = DatasetLoader(name="BNCI2014_001")
info = loader.info()
dataset = loader.read()  # raw 레코딩만

decoder = BraindecodeDecoder(
    "EEGNet",
    n_chans=info.metadata["channel_types"]["eeg"],
    n_outputs=len(info.metadata["event_ids"]),
    n_times=expected_window_samples(dataset, **WINDOWING_KWARGS),
    windowing_kwargs=WINDOWING_KWARGS,
)
model = decoder.model  # 실제 torch.nn.Module -- 아래에서 직접 forward를 확인해봅니다
print(model)

windows_dataset = decoder.prepare(
    dataset
)  # 전처리 + 윈도잉을 이 디코더의 설정 그대로 자동 적용

X0, y0, _ = windows_dataset[0]
LABEL_NAMES = label_names(
    windows_dataset
)  # 하드코딩하지 않고 windows_dataset 자신의 이벤트 매핑에서

n_chans, input_window_samples = X0.shape[0], X0.shape[1]
n_classes = len(set(windows_dataset.get_metadata()["target"]))
print(f"n_chans={n_chans}, n_times={input_window_samples}, n_classes={n_classes}")
print(f"총 윈도우 수: {len(windows_dataset)}")

### 이 윈도우가 실제로 어떤 모양인지 확인 — 디코더에 들어가기 직전의 데이터

히트맵(색으로 표현)과 그 밑에 **실제 숫자 행렬**을 같이 보여줍니다 — 히트맵의 색이 결국 이 숫자들이라는 걸 바로 확인할 수 있습니다. 이 윈도우는 정확히 큐 시작 시점(t=0)부터 시작합니다. 아래 1번 섹션에서 `EEGNet`에 넣는 것도 바로 이 `X0`입니다.


In [ ]:
print(f"이 샘플의 라벨: target={y0} -> '{LABEL_NAMES[y0]}'")

ch_names = dataset.datasets[0].raw.ch_names  # X0의 채널 순서와 동일한 22개 전극 이름
sfreq = dataset.datasets[0].raw.info["sfreq"]
times = np.arange(X0.shape[1]) / sfreq

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    X0,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-4,
    vmax=4,
    extent=[0, X0.shape[1] / sfreq, X0.shape[0], 0],
)
ax.set_yticks(np.arange(len(ch_names)) + 0.5)
ax.set_yticklabels(ch_names, fontsize=7)
ax.set_xlabel("time (s, 0 = window start = cue onset)")
ax.set_title(
    f"One window, all {len(ch_names)} channels, label='{LABEL_NAMES[y0]}'", pad=12
)
plt.colorbar(im, ax=ax, label="standardized amplitude")
plt.tight_layout()
plt.show()

# 위 히트맵과 완전히 같은 데이터를, 실제 숫자 행렬로도 봅니다 (앞/뒤만 보이게 자동 축약)
import pandas as pd

with pd.option_context(
    "display.max_rows",
    12,
    "display.max_columns",
    10,
    "display.float_format",
    lambda v: f"{v:.2f}",
):
    df = pd.DataFrame(X0, index=ch_names, columns=[f"{t:.2f}s" for t in times])
    display(df)

## 1. `EEGNet` 디코더 input/output shape 확인

위에서 이미 만든 `decoder`(`n2o.decoder.BraindecodeDecoder`, `braindecode.models`에서 이름으로 아키텍처를 찾아 만들어줍니다 -- `src/n2o/decoder/braindecode_entry.py`)를 그대로 써서, 실제로 forward pass가 어떤 shape을 흘려보내는지 확인합니다. `BraindecodeDecoder.list_models()`로 등록된 아키텍처 이름을 전부 볼 수 있습니다(`EEGNet`, `ShallowFBCSPNet`, ... 약 60개).


In [ ]:
device = next(model.parameters()).device
print("model device:", device)

model.eval()
with torch.no_grad():
    X_batch = torch.tensor(X0).unsqueeze(0).float().to(device)
    print("model input shape (batch, n_chans, n_times):", tuple(X_batch.shape))
    out = model(X_batch)
    print("model output shape (batch, n_classes):", tuple(out.shape))
    print("raw logits:", out)
    print(
        "note: 현재 버전은 마지막에 LogSoftmax가 없고 raw logit을 그대로 반환함 -> loss는 CrossEntropyLoss를 써야 함"
    )

In [ ]:
display(
    Markdown(
        "### 모델 출력을 그림으로 보기 — raw logit과 softmax 확률이 클래스별로 어떻게 나오는지 확인합니다."
    )
)

probs = torch.softmax(out, dim=1)[0].detach().cpu().numpy()
logits_np = out[0].detach().cpu().numpy()
pred_idx = int(logits_np.argmax())
colors = ["tab:orange" if i == pred_idx else "tab:blue" for i in range(4)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(LABEL_NAMES, logits_np, color=colors)
axes[0].set_title("raw logit")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(LABEL_NAMES, probs, color=colors)
axes[1].set_title("softmax probability")
axes[1].tick_params(axis="x", rotation=20)

fig.suptitle(
    f"Predicted: '{LABEL_NAMES[pred_idx]}' (true label: '{LABEL_NAMES[y0]}') "
    "- untrained model, close to random"
)
plt.tight_layout()
plt.show()

## 2. 사전학습된 체크포인트로 추론하기

공개된 체크포인트를 내려받아 추론 해봅니다 — braindecode 공식 예제([Basic Brain Decoding on EEG Data](https://braindecode.org/stable/auto_examples/model_building/plot_bcic_iv_2a_moabb_trial.html))가 Hugging Face에 올려둔 `ShallowFBCSPNet` 체크포인트(`braindecode/plot_bcic_iv_2a_moabb_trial`)입니다.

이 체크포인트는 **subject 3**로 학습되었고, 사전학습 체크포인트는 항상 그 체크포인트를 학습시킬 때 쓴 전처리/윈도잉과 정확히 같은 shape을 요구합니다 — 위 0번 섹션에서 이미 그 shape에 맞춰 정의해둔 `WINDOWING_KWARGS`(**큐 기준 -0.5초 ~ +4.0초, 22채널×1125시점**)를 여기서도 그대로 재사용합니다. `BraindecodeDecoder`를 만들 때 그 `windowing_kwargs`를 넘겨두면, `decoder.prepare(raw_dataset)` 호출 한 번으로 밴드패스+표준화 전처리와 윈도잉을 그 디코더가 학습될 때와 똑같이 자동으로 적용합니다.


In [ ]:
from braindecode.datasets import MOABBDataset

subject_id = 3  # 체크포인트가 실제로 학습된 subject
ckpt_decoder = BraindecodeDecoder(
    "ShallowFBCSPNet",
    n_chans=n_chans,
    n_outputs=n_classes,
    n_times=input_window_samples,  # 0번 섹션과 같은 WINDOWING_KWARGS이므로 같은 윈도우 길이
    final_conv_length="auto",
    windowing_kwargs=WINDOWING_KWARGS,
)

ckpt_dataset = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=[subject_id])
ckpt_windows = ckpt_decoder.prepare(
    ckpt_dataset
)  # 전처리 + 윈도잉을 이 디코더의 설정 그대로 자동 적용

X0_ckpt, y0_ckpt, _ = ckpt_windows[0]
print("체크포인트용 window shape:", X0_ckpt.shape)  # (22, 1125)

splitted = ckpt_windows.split("session")
valid_set = splitted["1test"]
print(f"평가용 trial 수: {len(valid_set)}")

In [ ]:
from braindecode import EEGClassifier
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # HF_TOKEN 미설정 안내 문구(X-HF-Warning) 숨기기

clf = EEGClassifier(
    ckpt_decoder.model,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.AdamW,
    device="cpu",
    classes=list(range(n_classes)),
)
clf.initialize()  # skorch가 내부 상태(옵티마이저 등)를 만들도록 -- 아직 학습은 안 함

repo_id = "braindecode/plot_bcic_iv_2a_moabb_trial"
clf.load_params(
    f_params=hf_hub_download(repo_id, "params.safetensors"),
    f_history=hf_hub_download(repo_id, "history.json"),
    use_safetensors=True,
)
print(f"{repo_id}에서 실제 학습된 가중치를 불러왔습니다.")

In [ ]:
y_true = valid_set.get_metadata().target.values
y_pred = clf.predict(valid_set)

accuracy = (y_true == y_pred).mean()
print(f"held-out(valid) 정확도: {accuracy:.1%}  (4-class 문제, 무작위 추측은 25%)")

per_class_acc = [
    (y_pred[y_true == c] == c).mean() if (y_true == c).any() else float("nan")
    for c in range(len(LABEL_NAMES))
]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(LABEL_NAMES, per_class_acc, color="tab:blue")
ax.axhline(
    accuracy,
    color="black",
    linestyle="--",
    linewidth=1,
    label=f"overall={accuracy:.1%}",
)
ax.axhline(0.25, color="tab:red", linestyle=":", linewidth=1, label="chance (25%)")
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1.0)
ax.set_title(f"Pretrained ShallowFBCSPNet, subject {subject_id} held-out session")
ax.legend()
plt.tight_layout()
plt.show()